# LoRA replication on Gemma 3 1B: baseball QA

Runs every experiment for the MSDS 534 project: base model (zero-shot and few-shot), LoRA, QLoRA,
full fine-tuning, then the rank and module comparisons from the paper (Tables 5 and 6).

**Before you start**
1. *Runtime → Change runtime type → **L4 GPU*** (use A100 only if full fine-tuning runs out of memory, and then rerun everything on A100 so numbers are comparable).
2. Colab *Secrets* (key icon, left sidebar): add `HF_TOKEN` (a Hugging Face read token from an account that has accepted the Gemma terms). If the GitHub repo is private, also add `GITHUB_TOKEN`.
3. Run top to bottom. Results are written to Google Drive, and finished runs are **skipped** on rerun, so a disconnect never loses work: reconnect and run all again.

The code lives in `src/lora_exp.py` (training, evaluation, measurement) and `src/scoring.py` (answer scoring).

## 0. Setup

In [ ]:
REPO_URL = "https://github.com/vatsalmalkari/LoRA-finetuning-project.git"
REPO_DIR = "/content/LoRA-finetuning-project"
DRIVE_DIR = "/content/drive/MyDrive/lora_project"   # results persist here

import os
from google.colab import drive, userdata
drive.mount("/content/drive")

def secret(name):
    try:
        return userdata.get(name)
    except Exception:
        return None

if not os.path.exists(REPO_DIR):
    gh = secret("GITHUB_TOKEN")
    url = REPO_URL.replace("https://", f"https://{gh}@") if gh else REPO_URL
    !git clone -q {url} {REPO_DIR}
else:
    !git -C {REPO_DIR} pull -q
%cd {REPO_DIR}
!git log --oneline -1

In [ ]:
!pip install -q -U transformers peft accelerate bitsandbytes

In [ ]:
import sys, json, torch
sys.path.insert(0, "src")
import importlib, lora_exp, scoring
importlib.reload(scoring); importlib.reload(lora_exp)
from lora_exp import RunConfig, run, load_jsonl, check_frozen_test, summarize, mcnemar, load_results

from huggingface_hub import login
login(secret("HF_TOKEN"))

import transformers, peft
print("torch", torch.__version__, "| transformers", transformers.__version__, "| peft", peft.__version__)
gpu = torch.cuda.get_device_name(0)
mem = torch.cuda.get_device_properties(0).total_memory / 2**30
# is_bf16_supported() is True even on T4 (emulated, slow); real bf16 needs compute capability 8.0+
PRECISION = "bf16" if torch.cuda.get_device_capability(0)[0] >= 8 else "fp32"
print(f"GPU: {gpu} ({mem:.1f} GB)   precision: {PRECISION}")
if "T4" in gpu:
    print("WARNING: T4 has no fast bf16 and Gemma 3 can overflow in fp16; using fp32. "
          "Use the T4 only for the memory demo in section 6.")

## 1. Data

`USE_UNVERIFIED = True` reads the pipeline-check split built before the rules questions were verified.
It is fine for section 2 but **never report numbers from it**. Once the team has verified
`data/raw/rules_mc.json` and run `python src/build_dataset.py`, commit `data/processed/`, push, and set this to `False`.

In [ ]:
USE_UNVERIFIED = False   # real, frozen split (rules verified 2026-11-24)

DATA_DIR = "data/processed/unverified" if USE_UNVERIFIED else "data/processed"
data = {s: load_jsonl(f"{DATA_DIR}/{s}.jsonl") for s in ["train", "val", "test"]}
if not USE_UNVERIFIED:
    print("frozen test set OK:", check_frozen_test(DATA_DIR)[:16])
RESULTS = f"{DRIVE_DIR}/{'results_unverified' if USE_UNVERIFIED else 'results'}"
print({s: len(v) for s, v in data.items()}, "->", RESULTS)
print(data["train"][0]["messages"])

### Model shapes (for the report's math section)

LoRA on a weight $W_0 \in \mathbb{R}^{d_{out} \times d_{in}}$ adds $r(d_{in} + d_{out})$ parameters.
The paper's count $2 \hat L d_{model} r$ assumes square $W_q, W_v$; Gemma 3 uses grouped-query attention, so they are not square.

In [ ]:
from transformers import AutoConfig
c = AutoConfig.from_pretrained(lora_exp.MODEL_ID)
c = getattr(c, "text_config", c)
d, L = c.hidden_size, c.num_hidden_layers
hd = getattr(c, "head_dim", d // c.num_attention_heads)
shapes = {"q_proj": (d, c.num_attention_heads * hd), "k_proj": (d, c.num_key_value_heads * hd),
          "v_proj": (d, c.num_key_value_heads * hd), "o_proj": (c.num_attention_heads * hd, d)}
print(f"d_model={d}  layers={L}  heads={c.num_attention_heads}  kv_heads={c.num_key_value_heads}  head_dim={hd}")
for k, (i, o) in shapes.items():
    print(f"  {k}: {i} -> {o}   LoRA params per layer per unit rank: {i + o}")
for targets, r in [(("q_proj", "v_proj"), 8), (("q_proj",), 8), (("q_proj", "v_proj"), 4),
                   (("q_proj", "k_proj", "v_proj", "o_proj"), 2)]:
    n = L * r * sum(sum(shapes[t]) for t in targets)
    print(f"  r={r} on {'+'.join(targets)}: {n:,} trainable params")

## 2. Pipeline check

LoRA on 20 training examples for one epoch, scored on the validation set. Confirm that the loss goes down,
nothing is NaN, and the responses look like answers. Written to a separate folder, never reported.

In [ ]:
chk = run(RunConfig("lora", n_train=20, epochs=1, batch_size=4, precision=PRECISION),
          data, f"{DRIVE_DIR}/pipeline_check", eval_split="val", latency=False, save_adapter=False,
          overwrite=True)
print("losses:", [round(x, 3) for x in chk["train"]["losses"]])
print(f"peak memory {chk['peak_train_mem_gb']:.2f} GB, {chk['train']['tokens_per_second']:.0f} tok/s")
preds = json.load(open(f"{DRIVE_DIR}/pipeline_check/{chk['name']}.predictions.json"))["zero_shot"]
for p in preds[:8]:
    print(f"[{p['qtype']}] gold={p['answer']!r:22} got={p['response'][:60]!r}  strict={p['strict']}")

## 3. Base model (zero-shot and few-shot)

Few-shot gives the model three solved examples of the same question type before each test question.
It is the fair comparison for LoRA, because it also shows the model the answer format.

In [ ]:
base = run(RunConfig("base", precision=PRECISION), data, RESULTS)
for k in ["accuracy", "accuracy_fewshot"]:
    print(k, {g: f"{v['strict']:.2f}/{v['lenient']:.2f}" for g, v in base[k].items()})

## 4. Learning-rate selection on the validation set (seed 1)

Pick each method's learning rate by **validation** strict accuracy, then fix it below.
Never pick anything by test accuracy. (30 validation items is noisy: break near-ties toward the paper's or common defaults: 2e-4 for LoRA, 2e-5 for full fine-tuning.)

In [ ]:
LR_GRID = {"lora": [1e-4, 2e-4, 5e-4], "full": [1e-5, 2e-5, 5e-5]}
val_rows = []
for method, lrs in LR_GRID.items():
    for lr in lrs:
        r = run(RunConfig(method, lr=lr, seed=1, precision=PRECISION), data, RESULTS,
                eval_split="val", latency=False, save_adapter=False)
        if not r.get("oom"):
            val_rows.append((method, lr, r["accuracy"]["all"]["strict"], r["accuracy"]["all"]["lenient"],
                             r["train"]["losses"][-1]))
import pandas as pd
pd.DataFrame(val_rows, columns=["method", "lr", "val_strict", "val_lenient", "final_loss"])

In [ ]:
# Set these from the table above.
LR = {"lora": 2e-4, "qlora": 2e-4, "full": 2e-5, "top2": 1e-4}

## 5. Main experiments: 3 seeds each

LoRA and QLoRA at rank 8 on $W_q, W_v$; full fine-tuning; and the last-two-layers baseline.
Each run trains on 300 examples for 3 epochs, scores the 100 test items, and times 64-token generation.
For LoRA it also times the merged model ($W = W_0 + BA$), which tests the paper's "no added latency" claim.

In [ ]:
SEEDS = [1, 100, 1000]
MAIN = ["lora", "qlora", "full", "top2"]
for seed in SEEDS:
    for method in MAIN:
        run(RunConfig(method, lr=LR[method], seed=seed, precision=PRECISION), data, RESULTS)

### Results

In [ ]:
pd.set_option("display.width", 250); pd.set_option("display.max_columns", 60)
table = summarize(RESULTS)
cols = ["n_seeds", "all_strict", "all_lenient", "mc_strict", "short_strict", "stats_strict",
        "trainable_params", "trainable_pct", "train_min", "tok_per_s", "peak_mem_gb",
        "checkpoint_mb", "latency_ms", "latency_merged_ms"]
table[[c for c in cols if c in table.columns.get_level_values(0)]].round(3)

### Paired tests on the same 100 test items

Seed-to-seed spread does not include test-set sampling error. McNemar's exact test compares two methods on the
*same* items: it only uses the items where one is right and the other wrong. Run per seed.

In [ ]:
def preds_of(name, setting="zero_shot"):
    return json.load(open(f"{RESULTS}/{name}.predictions.json"))[setting]

ref = preds_of("base", "few_shot")
rows = []
for r in load_results(RESULTS):
    if r["config"]["method"] == "base" or r.get("oom") or r["config"]["n_train"]:
        continue
    for metric in ["strict", "lenient"]:
        t = mcnemar(ref, preds_of(r["name"]), metric)
        rows.append({"run": r["name"], "metric": metric, "base_fewshot": t["acc_a"], "method": t["acc_b"],
                     "wins": t["b_right_a_wrong"], "losses": t["a_right_b_wrong"], "p": t["p_value"]})
pd.DataFrame(rows).sort_values(["metric", "run"]).round(3)

### Training loss curves

In [ ]:
import matplotlib.pyplot as plt
fig, ax = plt.subplots(figsize=(7, 4))
for r in sorted(load_results(RESULTS), key=lambda r: r["name"]):
    c = r["config"]
    if c["method"] in MAIN and c["seed"] == 1 and not r.get("oom") and not c["n_train"] and "train" in r:
        ax.plot(r["train"]["losses"], label=c["method"])
ax.set_xlabel("optimizer step"); ax.set_ylabel("training loss (answer tokens)"); ax.legend()
ax.set_title("Training loss, seed 1"); plt.show()

## 6. Paper Tables 5 and 6: which matrices, and what rank (after the first draft)

**Rank sweep** (Table 6): LoRA on $W_q, W_v$ with $r \in \{1, 2, 4, 8, 64\}$.
**Module comparison** (Table 5): a similar parameter budget spent on different matrices.
Seed 1 only; add seeds if time allows.

In [ ]:
for r_ in [1, 2, 4, 8, 64]:
    run(RunConfig("lora", rank=r_, alpha=2 * r_, lr=LR["lora"], seed=1, precision=PRECISION), data, RESULTS,
        latency=False)
for targets, r_ in [(("q_proj",), 8), (("v_proj",), 8), (("q_proj", "v_proj"), 4),
                    (("q_proj", "k_proj", "v_proj", "o_proj"), 2)]:
    run(RunConfig("lora", rank=r_, alpha=2 * r_, targets=targets, lr=LR["lora"], seed=1, precision=PRECISION),
        data, RESULTS, latency=False)
summarize(RESULTS).query("method == 'lora'")[["all_strict", "all_lenient", "trainable_params"]].round(3)

*Note on alpha:* the paper fixes $\alpha$ to the first $r$ tried and scales the update by $\alpha/r$. The main runs use
$\alpha = 16, r = 8$ (scale 2); the sweep keeps the scale at 2 ($\alpha = 2r$) so that only the rank changes. State this in the report.

## 7. Optional: memory demo on a free T4

Switch the runtime to **T4**, rerun sections 0 and 1, then run this cell. Full fine-tuning is expected to run out of
memory while LoRA fits, which is the paper's "lowers the hardware barrier" claim. OOMs are saved as results.

In [ ]:
assert "T4" in torch.cuda.get_device_name(0), "switch the runtime to T4 first"
for method in ["full", "lora"]:
    run(RunConfig(method, lr=LR[method], seed=1, precision="fp32", tag="_t4"), data, RESULTS, latency=False)